# 06 — SICAR: outcomes H1c (canal compliance)

Constrói os outcomes intermediários do canal compliance (H1c) a partir do painel SICAR mensal:

- **`cobertura_car_ativo`** = (Σ Imóvel Área onde Status=Ativo) / area_total_municipal_MB
- **`adesao_pra`** = (Σ área com PRA=Sim) / Σ área total cadastrada
- **`share_veg_nativa_atual`** = Σ Vegetação Nativa Atual / Σ Imóvel Área total
- **`share_pra_nao_informado`** (auxiliar) — fração de área com PRA = 'Não Informado'

**Decisões metodológicas confirmadas:**
- S1 (snapshot anual): dezembro de cada ano (último mês = abr/2026 com flag).
- S2 (denominador cobertura): `area_total_ha` do MapBiomas.
- S3 (status): Ativo estrito (sensibilidade `cobertura_car_ativo_pendente` em coluna separada).
- S4 (PRA Não Informado): denominador inclui todos; share_pra_nao_informado fica como auxiliar.
- Restrição: apenas `Tipo='Imóvel rural'` (descarta assentamento e povos tradicionais).

**Outputs em `data/interim/`:**
- `sicar_outcomes_anual.csv` — município × ano × outcomes H1c (~10k linhas)

**Pré-requisito:** rodar `05_mapbiomas.ipynb` antes (precisa de `mapbiomas_panel.csv` para o denominador).

**Tempo esperado:** ~30-40 segundos.

In [1]:
# Setup portável — resolve a raiz do repositório sem depender do Google Drive.
# Para executar a partir do Drive, defina antes: os.environ["RENOVABIO_BASE_DIR"] = "<caminho>"
# Para executar a partir do Drive, defina antes de rodar esta célula:
import os
import sys
from pathlib import Path

if os.environ.get("RENOVABIO_BASE_DIR"):
    BASE_DIR = Path(os.environ["RENOVABIO_BASE_DIR"]).expanduser().resolve()
else:
    BASE_DIR = Path.cwd().resolve()
    while not (BASE_DIR / "requirements.txt").exists() and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [2]:
# Reload módulos
import importlib
from pipeline import config, normalize, io, sicar
importlib.reload(config); importlib.reload(normalize)
importlib.reload(io); importlib.reload(sicar)

from pipeline.config import PARAMS, interim, out_pre
from pipeline.sicar import run_sicar_pipeline, SICAR_IBGE_FIXES
print('✓ módulos carregados')
print(f'  Correções ortográficas SICAR_IBGE_FIXES: {len(SICAR_IBGE_FIXES)} entradas')
for k, v in SICAR_IBGE_FIXES.items():
    print(f'    {k!r:35s} → {v!r}')

✓ módulos carregados
  Correções ortográficas SICAR_IBGE_FIXES: 4 entradas
    'DONA EUSEBIA|MG'                   → 'DONA EUZEBIA|MG'
    'SAO LUIS DO PARAITINGA|SP'         → 'SAO LUIZ DO PARAITINGA|SP'
    'SAO THOME DAS LETRAS|MG'           → 'SAO TOME DAS LETRAS|MG'
    'EMBU|SP'                           → 'EMBU DAS ARTES|SP'


In [3]:
# Carrega crosswalk e verifica que MapBiomas já rodou
cw = pd.read_csv(interim('crosswalk_centrosul.csv'), dtype={'geocode': str})
print(f'Crosswalk: {cw.shape}')

mb_path = interim('mapbiomas_panel.csv')
if not mb_path.exists():
    raise FileNotFoundError(
        '⚠️ mapbiomas_panel.csv não encontrado. Rode 05_mapbiomas.ipynb antes.'
    )
print(f'✓ mapbiomas_panel.csv encontrado ({mb_path.stat().st_size / 1024 / 1024:.1f} MB)')

Crosswalk: (2363, 5)
✓ mapbiomas_panel.csv encontrado (16.1 MB)


## Roda pipeline SICAR completo

In [ ]:
result = run_sicar_pipeline(crosswalk=cw, save=True)

→ Lendo painel SICAR (~30 MB, pode levar 30s)...
  raw: (344124, 14)

→ Filtrando Tipo='Imóvel rural'...
  filtered: (342035, 14)

→ Parsing município (UF - NOME → muni_key)...

→ Tomando snapshot anual (dezembro ou último mês)...
  snapshot: (39768, 21)

→ Restringindo à janela do painel (2012-2024)...
  in window: (22852, 21)
  anos disponíveis: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]

→ Resolvendo geocode via crosswalk (com SICAR_IBGE_FIXES)...
  matched: 22852 cells | unmatched: 0 cells

→ Restringindo ao Centro-Sul...
  CS: 22852 cells, 2283 munis

→ Agregando para (geocode × ano)...
  aggregated: (10191, 10)

→ Carregando painel MapBiomas para denominador (area_total_ha)...


## Inspeção dos outcomes H1c

In [ ]:
outcomes = result['outcomes']
print(f'Shape: {outcomes.shape}')
print(f'\nColunas:')
for c in outcomes.columns:
    print(f'  - {c}')
print(f'\nAmostra (município × ano):')
outcomes.head(5)

In [ ]:
# Cobertura temporal
print(f'Anos disponíveis: {sorted(outcomes["ano"].dropna().unique().tolist())}')
print(f'Total munis CS no SICAR: {outcomes["geocode"].nunique()}')
print(f'(Crosswalk tem 2.363; munis sem cadastro CAR ficam fora do painel SICAR)')
print()
print('Por UF (em 2024):')
print(outcomes[outcomes['ano'] == 2024].groupby('uf').size().to_string())

In [ ]:
# Estatísticas dos 5 outcomes em 2024
print('Distribuições dos outcomes em 2024:\n')
outcomes_2024 = outcomes[outcomes['ano'] == 2024]
cols = ['cobertura_car_ativo', 'cobertura_car_ativo_pendente',
        'adesao_pra', 'share_pra_nao_informado', 'share_veg_nativa_atual']
outcomes_2024[cols].describe()

In [ ]:
# Top 15 munis por cobertura_car_ativo em 2024
print('Top 15 munis por cobertura_car_ativo em 2024:')
top_cob = outcomes[outcomes['ano'] == 2024].nlargest(15, 'cobertura_car_ativo')
top_cob[['municipio', 'uf', 'cobertura_car_ativo', 'adesao_pra',
         'share_veg_nativa_atual']]

In [ ]:
# Trajetória temporal nacional dos outcomes (mean por ano)
import matplotlib.pyplot as plt

anos = sorted(outcomes['ano'].dropna().unique())
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, c in zip(axes, ['cobertura_car_ativo', 'adesao_pra', 'share_veg_nativa_atual']):
    means = outcomes.groupby('ano')[c].mean()
    ax.plot(means.index, means.values, marker='o')
    ax.set_title(c)
    ax.set_xlabel('Ano')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('\nTrajetórias agregadas — espera-se cobertura_car_ativo crescente (CAR foi se expandindo)')

## Resumo

Se o pipeline rodou com sucesso (~10k linhas, ~2.280 munis CS, anos 2014-2024), SICAR está validado.

**Outputs em `data/interim/`:**
- `sicar_outcomes_anual.csv` — input principal de `09_assembly.ipynb` para canais H1c

**Próximas camadas:**
- `07_psm_baseline.ipynb` — covariáveis socioeconômicas baseline (Censo Agro 2017, IDHM, etc.)
- `09_assembly.ipynb` — painel completo + filtro canavieiro (universo final do paper)